<a href="https://colab.research.google.com/github/belokonr/ECON5200-Applied-Data-Analytics-in-Economics/blob/main/econ-lab-19-random-forests/lab_ch19_diagnostic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 19: Tree-Based Models — Random Forests
## ECON 5200: Causal Machine Learning & Applied Analytics
### Diagnosis-First Lab | 30 min Core + 15 min Extension + SHAP Deep Dive

---

**Format:** This lab contains **deliberately flawed code and analysis**. Your job:
1. Run the code
2. Identify what is wrong (not told what to look for)
3. Fix the issue
4. Document your reasoning
5. Extend the corrected analysis

**Verification checkpoints** are provided so you can confirm you found the right error.

---

In [1]:
# -----------------------------------------------------------
# GUIDED — Run as-is
# Step 1: Import libraries and load data
# -----------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
data = fetch_california_housing()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

## Part 1: Find the Bug — Model Comparison (10 min)

The following code trains three models and reports their performance.
**Something is wrong with how the comparison is set up.** Find it, fix it, explain.

In [11]:
# -----------------------------------------------------------
# GUIDED — Run as-is (contains deliberate error)
# Step 2: Model comparison — find the bug
# -----------------------------------------------------------

tree = DecisionTreeRegressor(random_state=RANDOM_STATE)
tree.fit(X_train, y_train)

ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)

# BUG IS HERE: RF is evaluated on TRAINING data, not test data
rf = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE)
rf.fit(X_train, y_train)

print('=== Model Comparison ===')
print(f"Single Tree  \u2014 R\u00b2: {r2_score(y_test, tree.predict(X_test)):.4f}")
print(f"Ridge        \u2014 R\u00b2: {r2_score(y_test, ridge.predict(X_test)):.4f}")
print(f"Random Forest \u2014 R\u00b2: {r2_score(y_train, rf.predict(X_train)):.4f}")  # \u2190 WRONG: using training set
print()
print('Conclusion: Random Forest achieves R\u00b2 > 0.97! Far superior to alternatives.')

=== Model Comparison ===
Single Tree  — R²: 0.6221
Ridge        — R²: 0.5759
Random Forest — R²: 0.9736

Conclusion: Random Forest achieves R² > 0.97! Far superior to alternatives.


### YOUR DIAGNOSIS

1. **What is wrong?** (identify the specific line and error type)
2. **Why is this dangerous?** (what misleading conclusion does it lead to?)
3. **Fix the code below** and report the correct R²

**Verification checkpoint:** After fixing, the RF Test R² should be between 0.78 and 0.83. If you get >0.95, you haven't found the bug.

4. **Which chapter concept does this error violate?** (hint: Ch 15)

In [12]:
# -----------------------------------------------------------
# ✏️ YOUR TASK — Fill in the blanks
# Fix the model comparison bug from Part 1
# -----------------------------------------------------------

# YOUR FIX HERE
print(f"Random Forest — R²: {r2_score(y_test, rf.predict(X_test)):.4f}")


Random Forest — R²: 0.8051


The R^2 was too high before since we evaluated on training data, which concealed overfitting.

## Part 2: Find the Methodological Flaw — Feature Importance (10 min)

The following analysis uses feature importance to make a **causal claim**.
The code runs correctly. The methodology is wrong. Find the flaw.

In [4]:
# -----------------------------------------------------------
# GUIDED — Run as-is (contains methodological flaw)
# Step 3: Feature importance with flawed causal reasoning
# -----------------------------------------------------------

rf_correct = RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE)
rf_correct.fit(X_train, y_train)

importance = pd.Series(rf_correct.feature_importances_, index=X.columns).sort_values(ascending=False)
print('Feature Importance (MDI):')
print(importance.round(4))
print()
print('POLICY RECOMMENDATION:')
print(f'The top predictor is {importance.index[0]} (importance = {importance.iloc[0]:.3f}).')
print(f'Therefore, to increase housing prices, policymakers should focus on increasing {importance.index[0]}.')
print(f'The second most important lever is {importance.index[1]}.')

Feature Importance (MDI):
MedInc        0.5259
AveOccup      0.1381
Latitude      0.0886
Longitude     0.0883
HouseAge      0.0544
AveRooms      0.0444
Population    0.0307
AveBedrms     0.0296
dtype: float64

POLICY RECOMMENDATION:
The top predictor is MedInc (importance = 0.526).
Therefore, to increase housing prices, policymakers should focus on increasing MedInc.
The second most important lever is AveOccup.


### YOUR DIAGNOSIS

1. **What is the methodological flaw?** (the code is correct — the reasoning is wrong)
2. **Why can't we use MDI for policy recommendations?** (connect to Ch 10 DAGs and Ch 15 prediction vs. explanation)
3. **What would you need to make a causal claim?** (hint: Ch 24 DML)
4. **Bonus:** MDI has a known statistical bias. What is it, and what alternative would you use?

**Verification checkpoint:** Your diagnosis should mention at least: (a) prediction ≠ causation, (b) confounding/omitted variables, (c) MDI bias toward high-cardinality features.

In [13]:
# -----------------------------------------------------------
# ✏️ YOUR TASK — Fill in the blanks
# Run permutation importance and write a proper (non-causal)
# interpretation of the results
# -----------------------------------------------------------

# YOUR CORRECTED ANALYSIS HERE
# Use permutation importance instead of MDI
perm_result = permutation_importance(rf_correct, X_test, y_test, n_repeats=10, random_state=42)
perm_importance = pd.Series(perm_result.importances_mean, index=X.columns).sort_values(ascending=False)
print("Permutation Importance (unbiased):")
print(perm_importance.round(4))
print("\nNOTE: These features PREDICT housing prices. For CAUSAL claims, use DML (Ch 24).")


Permutation Importance (unbiased):
MedInc        0.7347
Latitude      0.4429
Longitude     0.3352
AveOccup      0.2036
HouseAge      0.0721
AveRooms      0.0271
AveBedrms     0.0095
Population    0.0087
dtype: float64

NOTE: These features PREDICT housing prices. For CAUSAL claims, use DML (Ch 24).


The interpretation is off because we have established correlation between MedInc and prices, not causation. There are other factors correlated with both that are the real driving factors, but their effects are concealed within MedInc. Using permutation importance instead gives a less biased ranking.

## Part 3: Hyperparameter Tuning + XGBoost Comparison (10 min)

Tune the RF, then compare against XGBoost (gradient boosting).

In [15]:
# -----------------------------------------------------------
# ✏️ YOUR TASK — Fill in the blanks
# Tune RF with GridSearchCV and compare with GBR
# -----------------------------------------------------------

from sklearn.ensemble import GradientBoostingRegressor

# 1. GridSearchCV on RF
grid_search = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid={'n_estimators': [100, 200, 500], 'max_depth': [10, 20, None], 'max_features': ['sqrt', 0.5]},
    cv=5, scoring='neg_mean_squared_error', n_jobs=-1
)
grid_search.fit(X_train, y_train)
best_rf = grid_search.best_estimator_

# 2. GradientBoosting
gbr = GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42)
gbr.fit(X_train, y_train)

# 3. Compare
for name, model in [('Ridge', ridge), ('RF (default)', rf), ('RF (tuned)', best_rf), ('GBR', gbr)]:
    rmse = np.sqrt(mean_squared_error(y_test, model.predict(X_test)))
    r2 = r2_score(y_test, model.predict(X_test))
    print(f"{name:20s} — RMSE: {rmse:.4f}, R²: {r2:.4f}")

# 1. GridSearchCV on RandomForestRegressor
# 2. Fit GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.1)
# 3. Compare Test RMSE and R\u00b2 for: Ridge, RF (default), RF (tuned), GBR
# 4. Which model wins? By how much? Is the difference practically significant?


KeyboardInterrupt: 

---

## Extension: SHAP Analysis (5200 depth — 15 min)

Use SHAP to explain individual predictions. Compare MDI ranking vs. SHAP ranking.

In [16]:
# -----------------------------------------------------------
# GUIDED — Run as-is
# Step 4: SHAP setup and TreeExplainer
# -----------------------------------------------------------

# Install SHAP if needed
!pip install shap

import shap

explainer = shap.TreeExplainer(best_rf)
shap_values = explainer.shap_values(X_test)

# Waterfall for observation 0 (high-value)
shap.plots.waterfall(shap.Explanation(
    values=shap_values[0], base_values=explainer.expected_value, data=X_test.iloc[0]
))

# Beeswarm (global)
shap.plots.beeswarm(shap.Explanation(
    values=shap_values, base_values=explainer.expected_value, data=X_test
))

# Create SHAP explainer for the tuned RF
# explainer = shap.TreeExplainer(best_rf)  # use your tuned RF from Part 3
# shap_values = explainer.shap_values(X_test)

# 1. Waterfall plot for 3 observations: one high-value, one low-value, one surprising
# shap.plots.waterfall(shap.Explanation(values=shap_values[0], base_values=explainer.expected_value, data=X_test.iloc[0]))

# 2. Beeswarm plot (global view)
# shap.plots.beeswarm(shap.Explanation(values=shap_values, base_values=explainer.expected_value, data=X_test))

# 3. Compare MDI ranking vs SHAP ranking \u2014 do they agree? Where do they diverge?

NameError: name 'best_rf' is not defined

### SHAP Interpretation (write as a .py module)

Create a reusable `shap_analysis.py` module with:
- `explain_prediction(model, X, idx)` → returns SHAP waterfall for observation `idx`
- `global_importance(model, X)` → returns SHAP beeswarm plot
- `compare_importance(model, X, y)` → returns side-by-side MDI vs SHAP ranking

Include docstrings and type hints. This is a portfolio artifact.

---
## AI-Assisted Expansion: SHAP Dashboard + Reusable Module

**The Generative AI Policy: Foundations First, Expansion Second.** You have now established manual mastery over decision trees, random forests, hyperparameter tuning, feature importance, and SHAP explanations. You are now authorized to operate under the "Co-Pilot Rule."

### Your Expansion Task (5200 — Advanced)
Build TWO artifacts:

**Artifact 1: `src/shap_utils.py` module** with:
- `explain_prediction(model, X, idx)` → SHAP waterfall plot
- `global_importance(model, X)` → SHAP beeswarm plot
- `compare_importance(model, X, y)` → side-by-side MDI vs SHAP ranking
- Full docstrings, type hints, and error handling

**Artifact 2: Interactive Streamlit app** that lets the user:
1. Adjust `n_estimators` (1-500) and `max_features` (1-8) with sliders
2. See SHAP waterfall + beeswarm plots update with each parameter change
3. Compare RF vs Ridge vs GBR performance as hyperparameters change
4. Toggle between MDI, permutation, and SHAP importance rankings

### P.R.I.M.E. Prompt
Copy and paste this into Claude or ChatGPT:

In [8]:
# -----------------------------------------------------------
# 🤖 AI EXPANSION — Co-Pilot required
# Copy the P.R.I.M.E. prompt above into Claude, then paste
# the generated code here. Run it and verify.
# -----------------------------------------------------------

# [Prep] Act as an expert Python Data Scientist specializing
# in SHAP explanations, interactive visualizations, and
# scikit-learn production workflows.
#
# [Request] I just completed a diagnosis-first lab where I
# compared Decision Trees, Ridge, Random Forests, and Gradient
# Boosting on California Housing data. I fixed evaluation bugs,
# diagnosed causal overclaiming from MDI, tuned hyperparameters
# with GridSearchCV, and generated SHAP waterfall + beeswarm
# plots. Now I need TWO artifacts:
#
# 1. A reusable `src/shap_utils.py` module with three functions:
#    - explain_prediction(model, X, idx) -> SHAP waterfall
#    - global_importance(model, X) -> SHAP beeswarm
#    - compare_importance(model, X, y) -> MDI vs SHAP side-by-side
#    Include type hints, docstrings, and error handling.
#
# 2. An interactive Plotly dashboard (or Streamlit app) with
#    ipywidgets sliders for n_estimators (1-500) and max_features
#    (1-8). The dashboard should update four panels:
#    (a) model comparison bar chart (RF vs Ridge vs GBR),
#    (b) SHAP beeswarm that updates with max_features,
#    (c) Train vs Test R\u00b2 as n_estimators increases,
#    (d) toggle between MDI / permutation / SHAP rankings.
#
# [Iterate] Use plotly.graph_objects, ipywidgets, shap, numpy,
# sklearn. Use the same variable names: X_train, X_test,
# y_train, y_test, data.feature_names. Do not use deprecated
# Plotly or SHAP functions.
#
# [Mechanism Check] Add inline comments explaining:
#   - How TreeExplainer differs from KernelExplainer
#   - Why SHAP values are additive (Shapley property)
#   - How ipywidgets observers trigger plot updates
#   - Why we re-fit inside the callback
#
# [Evaluate] Explain what the dashboard reveals about:
#   - The relationship between n_estimators, max_features,
#     and test performance
#   - Where MDI and SHAP rankings diverge and why
#   - The marginal value of additional trees beyond ~200

# PASTE AI-GENERATED CODE BELOW:
"""
shap_utils.py — Reusable SHAP explanation utilities for tree-based
and linear scikit-learn models on the California Housing dataset.

Key concepts embedded in this module:
- TreeExplainer vs KernelExplainer selection
- Shapley additivity property
- MDI vs SHAP importance comparison

Author: Data Science Lab
"""

from __future__ import annotations

import warnings
from typing import Optional, Union

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from sklearn.base import BaseEstimator, is_classifier
from sklearn.ensemble import (
    GradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.inspection import permutation_importance
from sklearn.tree import DecisionTreeRegressor

# ---------------------------------------------------------------------------
# Internal helpers
# ---------------------------------------------------------------------------

# Type alias for the tree models TreeExplainer supports natively
TreeModel = Union[
    DecisionTreeRegressor,
    RandomForestRegressor,
    GradientBoostingRegressor,
]


def _get_explainer(
    model: BaseEstimator,
    X: np.ndarray,
) -> shap.Explainer:
    """
    Select the right SHAP explainer for the model type.

    ── How TreeExplainer differs from KernelExplainer ──
    • TreeExplainer exploits the internal tree structure to compute
      *exact* Shapley values in O(TLD²) time, where T = #trees,
      L = #leaves, D = depth.  It is specific to tree-based models.
    • KernelExplainer is model-agnostic: it approximates Shapley values
      by sampling coalitions of features and fitting a weighted linear
      model (LIME-style), so it works on *any* predict function but
      is much slower and introduces sampling variance.

    We prefer TreeExplainer whenever the model is tree-based.
    """
    tree_types = (
        DecisionTreeRegressor,
        RandomForestRegressor,
        GradientBoostingRegressor,
    )
    if isinstance(model, tree_types):
        return shap.TreeExplainer(model)
    else:
        # Fall back to the model-agnostic explainer with a subsample
        # of the background data to keep runtime manageable.
        background = shap.sample(X, min(100, X.shape[0]))
        return shap.KernelExplainer(model.predict, background)


def _compute_shap_values(
    model: BaseEstimator,
    X: np.ndarray,
    feature_names: Optional[list[str]] = None,
) -> shap.Explanation:
    """
    Compute SHAP values and return a full shap.Explanation object.

    ── Why SHAP values are additive (Shapley property) ──
    For every prediction f(x), the Shapley values φ_j satisfy:
        f(x) = E[f(X)]  +  Σ_j  φ_j(x)
    This *additivity* (or "efficiency") axiom guarantees that the
    contributions of all features sum exactly to the difference
    between the model output and the expected (base) value.
    This is not an approximation — it is a mathematical identity
    inherited from cooperative game theory.
    """
    explainer = _get_explainer(model, X)
    shap_values = explainer(X)

    # Attach human-readable feature names if provided
    if feature_names is not None:
        shap_values.feature_names = feature_names

    return shap_values


# ---------------------------------------------------------------------------
# Public API
# ---------------------------------------------------------------------------


def explain_prediction(
    model: BaseEstimator,
    X: np.ndarray,
    idx: int,
    feature_names: Optional[list[str]] = None,
    max_display: int = 10,
) -> plt.Figure:
    """
    Generate a SHAP waterfall plot for a single observation.

    Parameters
    ----------
    model : fitted scikit-learn estimator
    X : array-like of shape (n_samples, n_features)
        The dataset (train or test) from which to pick the observation.
    idx : int
        Row index of the observation to explain.
    feature_names : list[str], optional
        Human-readable names for each feature.
    max_display : int
        Maximum number of features to show (default 10).

    Returns
    -------
    matplotlib.figure.Figure
        The waterfall plot figure (also displayed via plt.show).

    Raises
    ------
    IndexError
        If idx is out of bounds for X.
    """
    if idx < 0 or idx >= X.shape[0]:
        raise IndexError(
            f"idx={idx} is out of bounds for X with {X.shape[0]} rows."
        )

    shap_values = _compute_shap_values(model, X, feature_names)

    fig, ax = plt.subplots(figsize=(10, 6))
    plt.sca(ax)
    shap.plots.waterfall(shap_values[idx], max_display=max_display, show=False)
    plt.title(f"SHAP Waterfall — Observation {idx}", fontsize=13)
    plt.tight_layout()
    plt.show()
    return fig


def global_importance(
    model: BaseEstimator,
    X: np.ndarray,
    feature_names: Optional[list[str]] = None,
    max_display: int = 10,
) -> plt.Figure:
    """
    Generate a SHAP beeswarm plot showing global feature importance.

    The beeswarm arranges every observation's SHAP value for every
    feature on the x-axis, coloured by the feature value.  This
    reveals not just *which* features matter but *how* they matter
    (direction + magnitude + interaction spread).

    Parameters
    ----------
    model : fitted estimator
    X : array-like (n_samples, n_features)
    feature_names : list[str], optional
    max_display : int

    Returns
    -------
    matplotlib.figure.Figure
    """
    shap_values = _compute_shap_values(model, X, feature_names)

    fig, ax = plt.subplots(figsize=(10, 7))
    plt.sca(ax)
    shap.plots.beeswarm(shap_values, max_display=max_display, show=False)
    plt.title("SHAP Beeswarm — Global Feature Importance", fontsize=13)
    plt.tight_layout()
    plt.show()
    return fig


def compare_importance(
    model: BaseEstimator,
    X: np.ndarray,
    y: np.ndarray,
    feature_names: Optional[list[str]] = None,
    n_repeats: int = 10,
    random_state: int = 42,
) -> pd.DataFrame:
    """
    Side-by-side comparison of MDI, Permutation, and SHAP importance.

    MDI (Mean Decrease in Impurity) is fast but biased toward
    high-cardinality / noisy features because splits on such
    features reduce impurity a lot even when they don't generalise.
    SHAP importance (mean |φ_j|) is grounded in game theory and
    is additive + consistent.  Permutation importance measures the
    drop in test-set performance when a feature is shuffled, giving
    a model-agnostic, unbiased ranking.

    Parameters
    ----------
    model : fitted tree-based estimator (must expose feature_importances_)
    X, y : test data for permutation importance
    feature_names : list[str], optional
    n_repeats : int – repeats for permutation importance
    random_state : int

    Returns
    -------
    pd.DataFrame
        Columns: feature, mdi_rank, perm_rank, shap_rank,
                 mdi_value, perm_value, shap_value

    Also displays a grouped bar chart comparing the three methods.
    """
    if not hasattr(model, "feature_importances_"):
        raise AttributeError(
            "model must expose `feature_importances_` (tree-based). "
            "Got: " + type(model).__name__
        )

    names = (
        list(feature_names)
        if feature_names is not None
        else [f"x{i}" for i in range(X.shape[1])]
    )

    # 1. MDI importance
    mdi = model.feature_importances_

    # 2. Permutation importance (on the supplied X, y — should be test set)
    perm = permutation_importance(
        model, X, y, n_repeats=n_repeats, random_state=random_state
    )
    perm_mean = perm.importances_mean

    # 3. SHAP importance = mean |SHAP value| per feature
    shap_values = _compute_shap_values(model, X, names)
    shap_imp = np.abs(shap_values.values).mean(axis=0)

    # Build comparison DataFrame
    df = pd.DataFrame(
        {
            "feature": names,
            "mdi_value": mdi,
            "perm_value": perm_mean,
            "shap_value": shap_imp,
        }
    )
    # Rank (1 = most important)
    for col in ["mdi_value", "perm_value", "shap_value"]:
        df[col.replace("value", "rank")] = df[col].rank(ascending=False).astype(int)

    df = df[
        [
            "feature",
            "mdi_rank",
            "perm_rank",
            "shap_rank",
            "mdi_value",
            "perm_value",
            "shap_value",
        ]
    ]

    # ── Visualisation: grouped horizontal bar chart ──
    fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
    methods = [
        ("mdi_value", "MDI (Gini)", "#e07b54"),
        ("perm_value", "Permutation", "#54a0e0"),
        ("shap_value", "SHAP mean(|φ|)", "#6cc070"),
    ]
    order = df.sort_values("shap_value", ascending=True)

    for ax, (col, title, colour) in zip(axes, methods):
        ax.barh(order["feature"], order[col], color=colour, edgecolor="white")
        ax.set_title(title, fontsize=12, fontweight="bold")
        ax.set_xlabel("Importance")

    plt.suptitle(
        "Feature Importance: MDI vs Permutation vs SHAP",
        fontsize=14,
        fontweight="bold",
        y=1.02,
    )
    plt.tight_layout()
    plt.show()

    return df


In [ ]:
"""
interactive_dashboard.py — Four-panel interactive dashboard for
exploring Random Forest / Gradient Boosting hyperparameters on
California Housing via Plotly + ipywidgets.

Panels
------
(a) Model comparison bar chart  (RF vs Ridge vs GBR)
(b) SHAP beeswarm that updates with max_features
(c) Train vs Test R² as n_estimators increases
(d) Toggle between MDI / Permutation / SHAP rankings

Run in a Jupyter notebook:
    %run interactive_dashboard.py
or:
    from interactive_dashboard import build_dashboard
    build_dashboard()

Requirements: plotly, ipywidgets, shap, numpy, sklearn
"""

from __future__ import annotations

import warnings
from functools import lru_cache

import numpy as np
import plotly.graph_objects as go
from IPython.display import display, clear_output
import ipywidgets as widgets
import matplotlib
matplotlib.use("agg")          # non-interactive backend for inline SHAP plots
import matplotlib.pyplot as plt
import shap

from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore", category=FutureWarning)

# ──────────────────────────────────────────────────────────────
# Data preparation  (same variable names as the lab)
# ──────────────────────────────────────────────────────────────
data = fetch_california_housing()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.2, random_state=42
)


# ──────────────────────────────────────────────────────────────
# Widget definitions
# ──────────────────────────────────────────────────────────────
# ── How ipywidgets observers trigger plot updates ──
# Each widget exposes an `.observe(callback, names='value')` method.
# When the user drags the slider, ipywidgets fires `callback`
# with a change dict containing {'new': <value>}.  We attach a
# single `on_update` handler to both sliders so that moving
# *either* one re-fits the models and refreshes all four panels.

n_estimators_slider = widgets.IntSlider(
    value=100, min=10, max=500, step=10,
    description="n_estimators",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="420px"),
)

max_features_slider = widgets.IntSlider(
    value=8, min=1, max=8, step=1,
    description="max_features",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="420px"),
)

importance_toggle = widgets.ToggleButtons(
    options=["MDI", "Permutation", "SHAP"],
    value="SHAP",
    description="Ranking:",
    style={"description_width": "initial"},
)

# Output areas for each panel
out_comparison = widgets.Output()
out_beeswarm = widgets.Output()
out_learning = widgets.Output()
out_ranking = widgets.Output()


# ──────────────────────────────────────────────────────────────
# Panel (c) helper — learning curve over n_estimators
# ──────────────────────────────────────────────────────────────
def _learning_curve_data(max_features: int, max_n: int = 500, step: int = 25):
    """
    Fit RF at several n_estimators values and record train/test R².
    Uses warm_start to avoid redundant computation.
    """
    ns = list(range(step, max_n + 1, step))
    train_scores, test_scores = [], []

    rf = RandomForestRegressor(
        n_estimators=step, max_features=max_features,
        warm_start=True, random_state=42, n_jobs=-1,
    )

    for n in ns:
        rf.set_params(n_estimators=n)
        rf.fit(X_train, y_train)
        train_scores.append(r2_score(y_train, rf.predict(X_train)))
        test_scores.append(r2_score(y_test, rf.predict(X_test)))

    return ns, train_scores, test_scores


# ──────────────────────────────────────────────────────────────
# Master update callback
# ──────────────────────────────────────────────────────────────
def on_update(change=None):
    """
    Re-fit models with current slider values and refresh all panels.

    ── Why we re-fit inside the callback ──
    Hyperparameters like n_estimators and max_features change the
    model structure, so we *must* retrain to reflect the new
    configuration.  Caching is impractical here because each
    (n_estimators, max_features) pair yields a different model.
    """
    n_est = n_estimators_slider.value
    max_feat = max_features_slider.value
    method = importance_toggle.value

    # --- Fit the three models ---
    rf = RandomForestRegressor(
        n_estimators=n_est, max_features=max_feat,
        random_state=42, n_jobs=-1,
    )
    gbr = GradientBoostingRegressor(
        n_estimators=n_est, max_features=max_feat,
        random_state=42,
    )
    ridge = Ridge(alpha=1.0)

    rf.fit(X_train, y_train)
    gbr.fit(X_train, y_train)
    ridge.fit(X_train, y_train)

    r2_rf = r2_score(y_test, rf.predict(X_test))
    r2_gbr = r2_score(y_test, gbr.predict(X_test))
    r2_ridge = r2_score(y_test, ridge.predict(X_test))

    # ── Panel (a): Model comparison bar chart ──
    with out_comparison:
        clear_output(wait=True)
        fig_a = go.Figure(
            go.Bar(
                x=["Random Forest", "Gradient Boosting", "Ridge"],
                y=[r2_rf, r2_gbr, r2_ridge],
                marker_color=["#4C78A8", "#F58518", "#72B7B2"],
                text=[f"{v:.4f}" for v in [r2_rf, r2_gbr, r2_ridge]],
                textposition="outside",
            )
        )
        fig_a.update_layout(
            title=f"Test R² — n_est={n_est}, max_feat={max_feat}",
            yaxis=dict(title="R² Score", range=[0, 1]),
            template="plotly_white",
            height=370, margin=dict(t=50, b=40),
        )
        fig_a.show()

    # ── Panel (b): SHAP beeswarm (RF) ──
    # TreeExplainer computes exact Shapley values using the
    # tree structure — O(TLD²) — without any sampling.
    with out_beeswarm:
        clear_output(wait=True)
        explainer = shap.TreeExplainer(rf)
        # Use a subsample for speed in the interactive loop
        X_sample = X_test[:300]
        shap_values = explainer(X_sample)
        shap_values.feature_names = list(data.feature_names)

        fig_b, ax = plt.subplots(figsize=(8, 5))
        plt.sca(ax)
        shap.plots.beeswarm(shap_values, max_display=max_feat, show=False)
        plt.title(f"SHAP Beeswarm (RF) — max_features={max_feat}")
        plt.tight_layout()
        plt.show()

    # ── Panel (c): Learning curve — Train vs Test R² ──
    with out_learning:
        clear_output(wait=True)
        ns, train_r2, test_r2 = _learning_curve_data(max_feat)

        fig_c = go.Figure()
        fig_c.add_trace(go.Scatter(
            x=ns, y=train_r2, mode="lines+markers", name="Train R²",
            line=dict(color="#4C78A8", width=2),
        ))
        fig_c.add_trace(go.Scatter(
            x=ns, y=test_r2, mode="lines+markers", name="Test R²",
            line=dict(color="#F58518", width=2),
        ))
        # Vertical line at current n_estimators
        fig_c.add_vline(x=n_est, line_dash="dash", line_color="grey",
                        annotation_text=f"current ({n_est})")
        fig_c.update_layout(
            title=f"RF Learning Curve — max_features={max_feat}",
            xaxis_title="n_estimators",
            yaxis_title="R² Score",
            template="plotly_white",
            height=370, margin=dict(t=50, b=40),
        )
        fig_c.show()

    # ── Panel (d): Importance ranking toggle ──
    with out_ranking:
        clear_output(wait=True)

        names = list(data.feature_names)

        if method == "MDI":
            # MDI = model.feature_importances_ (mean impurity decrease)
            vals = rf.feature_importances_
            colour = "#e07b54"
            subtitle = "MDI (Gini impurity decrease)"
        elif method == "Permutation":
            perm = permutation_importance(
                rf, X_test, y_test, n_repeats=10, random_state=42
            )
            vals = perm.importances_mean
            colour = "#54a0e0"
            subtitle = "Permutation (test-set R² drop)"
        else:  # SHAP
            # Reuse explainer from panel (b)
            vals = np.abs(shap_values.values).mean(axis=0)
            colour = "#6cc070"
            # ── Shapley additivity reminder ──
            # Σ φ_j(x) + E[f(X)] = f(x) for every x.
            # mean(|φ_j|) is a principled importance measure because
            # it respects this additive decomposition.
            subtitle = "SHAP mean(|φ|)"

        # Sort descending
        order = np.argsort(vals)
        fig_d = go.Figure(
            go.Bar(
                y=[names[i] for i in order],
                x=vals[order],
                orientation="h",
                marker_color=colour,
            )
        )
        fig_d.update_layout(
            title=f"Feature Importance — {subtitle}",
            xaxis_title="Importance",
            template="plotly_white",
            height=370, margin=dict(t=50, b=40, l=100),
        )
        fig_d.show()


# ──────────────────────────────────────────────────────────────
# Wire observers and build layout
# ──────────────────────────────────────────────────────────────
def build_dashboard():
    """Assemble and display the four-panel interactive dashboard."""

    # ── How ipywidgets observers trigger plot updates ──
    # `.observe` registers a callback that fires whenever the
    # widget's `value` trait changes.  The callback receives a
    # dict with keys 'old', 'new', 'owner', 'name', 'type'.
    # We ignore the dict and just re-render everything.
    n_estimators_slider.observe(on_update, names="value")
    max_features_slider.observe(on_update, names="value")
    importance_toggle.observe(on_update, names="value")

    controls = widgets.VBox([
        widgets.HTML("<h2 style='margin:0'>🌲 California Housing — Hyperparameter Explorer</h2>"),
        widgets.HTML("<p style='color:#666; margin-top:4px'>"
                     "Drag the sliders to re-fit RF & GBR and watch all panels update.</p>"),
        widgets.HBox([n_estimators_slider, max_features_slider]),
        importance_toggle,
    ])

    top_row = widgets.HBox(
        [out_comparison, out_beeswarm],
        layout=widgets.Layout(width="100%"),
    )
    bottom_row = widgets.HBox(
        [out_learning, out_ranking],
        layout=widgets.Layout(width="100%"),
    )

    dashboard = widgets.VBox([controls, top_row, bottom_row])
    display(dashboard)

    # Initial render
    on_update()


# Auto-launch when run as script / %run
if __name__ == "__main__":
    build_dashboard()

---
## Digital Portfolio: Institutional Signaling

### Generate Your Professional README
Copy and paste the prompt below into Claude or ChatGPT. **Do NOT ask the AI to write Python code — only documentation.**

In [9]:
# -----------------------------------------------------------
# 🤖 AI EXPANSION — README generation (no code, just docs)
# -----------------------------------------------------------

# PASTE THIS PROMPT INTO CLAUDE:
#
# "I need help writing a project description for my data science lab.
# **Important Rule:** Do NOT generate any Python code for me.
#
# **What I did in this lab:**
# * Compared Decision Tree, Ridge Regression, and Random Forest on
#   California Housing data (20,640 observations, 8 features)
# * Tuned RF hyperparameters with GridSearchCV (n_estimators, max_depth,
#   max_features)
# * Extracted and compared MDI vs permutation feature importance
# * Built an RF classifier and compared AUC against logistic regression
# * Created an interactive dashboard with Plotly + ipywidgets
# * Key finding: RF achieved R\u00b2 = [YOUR VALUE] vs Ridge R\u00b2 = [YOUR VALUE]
#
# **Please write a README.md entry including:**
# 1. Project Title: Tree-Based Models \u2014 Random Forests
# 2. Objective: A professional one-sentence summary
# 3. Methodology: Bullet points of technical steps
# 4. Key Findings: Summary of results
# Make this sound like a professional tech economist wrote it."

### Push to GitHub

```bash
cd econ-lab-19-random-forests
git add notebooks/ figures/ README.md verification-log.md
git commit -m "Lab 19: Random Forest vs OLS — California Housing"
git push origin main
```

Submit your GitHub repo link on Canvas.